In [100]:
import numpy as np
import pandas as pd
import numpy as np
import heapq

from MetricsRecorder import MetricsRecorder
from MissionState import MissionState
from Policy import FIFOPolicy, Action
from SatelliteImage import ImageRepresentation, ImageStatus, SatelliteImage
from ScoringConfig import ScoringConfig, ScientificCategory
from SimulationConfig import SimulationConfig
from SimUtility import generate_capture_times, generate_contact_times, create_image, set_image_representation

In [101]:
def make_decision_groups(state : MissionState, scoring: ScoringConfig, group_id : int, contact_end_time : int):
    contact_candidates = []
    
    for candidate_image in state.onboard_images:
        if candidate_image.status != ImageStatus.STORED:
            continue
        for i in range(int(candidate_image.representation) + 1):
            pred_value = candidate_image.predicted_value * scoring.representation_multiplier(ImageRepresentation(i))
            size = candidate_image.representation_sizes[i]
            
            remaining_seconds = (contact_end_time - state.current_time) * SECONDS_PER_DAY
            transmission_seconds = (size * 8 / config.bandwidth)
            if transmission_seconds > remaining_seconds:
               continue 
            action = i + 1
            contact_candidates.append([candidate_image.image_id, pred_value, size, action])

    contact_candidates.append([None, 0, 0, 0])

    decision_group = {
        "group_id" : group_id, 
        "remaining_contact_days" : contact_end_time - state.current_time,
        "onboard_image_count" : len(state.onboard_images),
        "candidates" : contact_candidates
    }
    return decision_group

In [102]:
SECONDS_PER_DAY = 86400

def run_simulation(state: MissionState,
                   events,
                   config: SimulationConfig,
                   policy,
                   scoring: ScoringConfig,
                   recorder: MetricsRecorder,
                   rng,
                   generate_groups: bool,
                   roll_out: tuple[int, int, int],
                   group_id: int,
                   image_id: int,
                   record_state: int):
    """Run until a requested decision or the end of the mission."""
    if generate_groups:
        decision_groups = []

    if state is None:
        state = MissionState(config)

    contact_duration_days = config.contact_duration / 1440

    while events:
        event = events[0]
        event_type = event[1]

        # Leave the unprocessed event in the heap so a resumed call can handle it.
        if event_type in (1, 2) and record_state is not None and record_state == group_id:
            return state, recorder, events, image_id, group_id

        event = heapq.heappop(events)
        state.current_time = event[0]

        match event_type:
            case 0:  # Capture
                recorder.record_capture()
                image = create_image(image_id, event[0], scoring, rng)
                image_id += 1

                if image.original_size > state.available_storage_mb:
                    storage_plan = policy.on_storage_pressure(image, state)
                    action = storage_plan.incoming_action

                    if action == Action.DELETE:
                        image.status = ImageStatus.DELETED
                        recorder.record_final_image(image)
                        continue

                    set_image_representation(image, action)
                    for target, action in storage_plan.onboard_actions:
                        if action == Action.DELETE:
                            target.status = ImageStatus.DELETED
                            state.onboard_images.remove(target)
                            recorder.record_final_image(target)
                        else:
                            set_image_representation(target, action)
                else:
                    action = policy.on_image_capture(image, state)
                    set_image_representation(image, action)

                state.onboard_images.append(image)
                assert not state.storage_is_over_capacity

            case 1:  # Contact start
                recorder.record_contact()
                contact_end_time = event[0] + contact_duration_days

            case 2:  # Transmission end
                delivered_image = event[3]
                contact_end_time = event[4]

                delivered_image.status = ImageStatus.FULLY_DELIVERED
                delivered_image.downlink_time = event[0]
                state.onboard_images.remove(delivered_image)

                sent_value = scoring.delivered_value(
                    delivered_image.true_category, delivered_image.representation
                )
                recorder.record_final_image(
                    delivered_image, delivered_image.stored_size_mb, sent_value
                )

        if event_type in (1, 2):
            if generate_groups:
                decision_groups.append(
                    make_decision_groups(state, scoring, group_id, contact_end_time)
                )

            if roll_out is not None and roll_out[0] == group_id:
                action = Action(roll_out[2])
                if action != Action.SKIP:
                    image = next(
                        img for img in state.onboard_images
                        if img.image_id == roll_out[1]
                    )
                    size = image.representation_sizes[int(action) - 1]
                    transmission_seconds = size * 8 / config.bandwidth
                    transmission_end_time = (
                        state.current_time + transmission_seconds / SECONDS_PER_DAY
                    )
            else:
                image, action, transmission_end_time = policy.on_contact_available(
                    contact_end_time, state, config
                )

            if action != Action.SKIP:
                set_image_representation(image, action)
                image.status = ImageStatus.TRANSMITTING
                heapq.heappush(
                    events,
                    (transmission_end_time, 2, "transmission_end", image, contact_end_time),
                )

            group_id += 1

    for image in state.onboard_images:
        recorder.record_final_image(image)
    recorder.finish_run()

    if generate_groups:
        return decision_groups

    if roll_out is not None:
        records = recorder.final_image_records
        total_available_value = float(records["true_value"].sum())
        total_delivered_value = float(records["delivered_value"].sum())
        return total_available_value - total_delivered_value

In [104]:
import copy
from tqdm import tqdm

config = SimulationConfig()
config.rand_seed = 40
config.bandwidth = 5
config.mission_duration = 100
scoring = ScoringConfig()
policy = FIFOPolicy()
rng = np.random.default_rng(config.rand_seed)

capture_times = generate_capture_times(config, rng)
contact_times = generate_contact_times(config)
events = [(time, 0, "capture") for time in capture_times]
events += [(time, 1, "contact_start") for time in contact_times]
heapq.heapify(events)

recorder = MetricsRecorder(
    policy_name=policy.name,
    random_seed=config.rand_seed,
    available_downlink_mb_per_contact=config.bandwidth * config.contact_duration * 60 / 8,
)

group_events, group_rng, group_policy, group_recorder = copy.deepcopy(
    (events, rng, policy, recorder)
)
groups = run_simulation(
    None, group_events, config, group_policy, scoring, group_recorder, group_rng,
    True, None, 0, 0, None,
)

state = MissionState(config)
group_id = 0
image_id = 0

for group in tqdm(groups, desc="Groups"):
    state, recorder, events, image_id, group_id = run_simulation(
        state, events, config, policy, scoring, recorder, rng,
        False, None, group_id, image_id, group["group_id"],
    )

    for candidate in group["candidates"]:
        trial_state, trial_events, trial_recorder, trial_rng, trial_policy = copy.deepcopy(
            (state, events, recorder, rng, policy)
        )
        loss = run_simulation(
            trial_state, trial_events, config, trial_policy, scoring,
            trial_recorder, trial_rng, False,
            (group["group_id"], candidate[0], candidate[3]),
            group_id, image_id, None,
        )
        candidate.append(loss)


Groups: 100%|███████████████████████████████| 1392/1392 [37:57<00:00,  1.64s/it]


In [123]:
values = [candidate[4] for group in groups for candidate in group["candidates"]]
print(max(values))
print(min(values))

324.25
290.25


In [124]:
from dataclasses import asdict

def groups_to_columns(groups, config, scoring, policy, mission_id, simulator_version):
    """Flatten one mission's candidate groups for pd.DataFrame(...).to_parquet(...)."""
    metadata = {
        "mission_id": mission_id,
        "simulator_version": simulator_version,
        "policy": policy.name,
        **{f"config_{key}": value for key, value in asdict(config).items()},
        **{f"scoring_{key}": value for key, value in asdict(scoring).items()},
    }
    rows = []
    for group in groups:
        for candidate in group["candidates"]:
            rows.append({
                **metadata,
                "group_id": group["group_id"],
                "remaining_contact_days": group["remaining_contact_days"],
                "onboard_image_count": group["onboard_image_count"],
                "image_id": candidate[0],
                "predicted_value": candidate[1],
                "size_mb": candidate[2],
                "action": candidate[3],
                "rollout_loss": candidate[4] if len(candidate) > 4 else None,
            })
    return {key: [row[key] for row in rows] for key in rows[0]} if rows else {}


# Call after rollout labels have been appended to candidates:
# columns = groups_to_columns(groups, config, scoring, policy, "mission_40", "v1")
# pd.DataFrame(columns).to_parquet("candidates.parquet", index=False)


In [125]:
columns = groups_to_columns(groups, config, scoring, policy, "mission_1", "v1")

In [127]:
df = pd.DataFrame(columns)

In [128]:
df

,mission_id,simulator_version,policy,config_mission_duration,config_storage_capacity,config_image_arrival_rate,config_const_take_image,config_contact_frequency,config_contact_duration,config_bandwidth,...,scoring_compressed_multiplier,scoring_full_multiplier,group_id,remaining_contact_days,onboard_image_count,image_id,predicted_value,size_mb,action,rollout_loss
0,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,0,0.006944,1,0.0,0.540685,6.640574,1,308.50
1,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,0,0.006944,1,0.0,1.622056,77.473367,2,307.50
2,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,0,0.006944,1,0.0,2.162741,221.352477,3,307.00
3,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,0,0.006944,1,NaN,0.000000,0.000000,0,309.00
4,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,1,0.002845,0,NaN,0.000000,0.000000,0,307.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123103,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,1391,0.002663,44,998.0,0.412959,8.210320,1,306.50
123104,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,1391,0.002663,44,998.0,1.238878,95.787070,2,305.50
123105,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,1391,0.002663,44,999.0,0.441107,7.650299,1,306.75
123106,mission_1,v1,fifo,100,10.0,10.0,True,6,10.0,5,...,0.75,1.0,1391,0.002663,44,999.0,1.323320,89.253486,2,306.25


In [130]:
df.to_parquet("candidates2.parquet", index=False)

In [133]:
mid = len(df) // 2

middle_df = df.iloc[mid - 20 : mid + 20]
print(middle_df)

NameError: name 'mid' is not defined